# ECR

Notebook d'exécution pour le code de l'article **RecSys 2024** (`src_emo/`). Fork adaptée aux GPU RTX 5090 (CUDA 12.8, 32 Go de VRAM).
Chaque étape du README est séparée en cellule pour exécuter dans l'ordre souhaité ou relancer une seule étape.
Les cellules supposent un environnement créé avec `uv` dans `./.venv/`.

**Sections**

1. Préparation, vérification rapide et téléchargement des jeux de données
2. Tâche A — Fusion sémantique émotionnelle (`train_pre.py`)
3. Tâche B — Recommandation d'items sensible aux émotions (`train_rec.py`)
4. Tâche C — Génération de réponses alignées émotionnellement (`train_emp.py` + `infer_emp.py`)

Chaque étape d'entraînement a une variante **smoke-test** (5 steps, batch 2) juste avant la commande complète.
Commencez par la variante smoke-test quand vous itérez.

## 1. Préparation et vérification rapide

### Téléchargement des jeux de données et des checkpoints

1. Téléchargez l'archive des données ECR depuis Google Drive :  
   https://drive.google.com/file/d/1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_/view?usp=sharing
2. Décompressez son contenu dans `src_emo/data/emo_data/`.
3. Téléchargez les checkpoints publiés :  
   https://drive.google.com/file/d/1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19/view?usp=sharing
4. Décompressez leur contenu dans `src_emo/data/saved/`.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/home/victor/projet/ECR").resolve()
SRC = PROJECT_ROOT / "src_emo"

os.environ["HF_HOME"] = str(PROJECT_ROOT / ".hf_cache")
os.environ["TRANSFORMERS_CACHE"] = str(PROJECT_ROOT / ".hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["SRC"] = str(SRC)
os.environ["MODEL_ARGS"] = (
    "--tokenizer microsoft/DialoGPT-small "
    "--text_tokenizer roberta-base "
    "--model microsoft/DialoGPT-small "
    "--text_encoder roberta-base"
)

print("project root :", PROJECT_ROOT)
print("src dir      :", SRC)
print("HF cache     :", os.environ["HF_HOME"])

In [ ]:
import torch, transformers, accelerate, torch_geometric

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate  :", accelerate.__version__)
print("pyg         :", torch_geometric.__version__)
print("cuda avail  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device      :", torch.cuda.get_device_name(0))
    print("compute cap :", torch.cuda.get_device_capability(0))

In [ ]:
for p in [
    SRC / "save" / "redial_rec",
    SRC / "save" / "redial_conv",
    SRC / "save" / "redial_gen",
    SRC / "save" / "redial_emp",
    SRC / "data" / "saved" / "pre-trained",
    SRC / "data" / "saved" / "rec-trained",
    SRC / "data" / "saved" / "emp-trained",
]:
    p.mkdir(parents=True, exist_ok=True)
    print("ok:", p.relative_to(PROJECT_ROOT))

## 2. Tâche A — Sous-tâche de fusion sémantique émotionnelle (`train_pre.py`)

Prétraitement pour redial, puis pré-entraînement de l'encodeur KG-prompt.
Correspond au premier bloc de `README.md`.

### 2.1  Prétraitement : copier `emo_data/` dans `redial/` puis lancer `process.py`

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
cp -r data/emo_data/* data/redial/
cd data/redial && python process.py
""",
    ],
    check=True,
)

### 2.2  Smoke-test (5 étapes, batch 2) — échoue rapidement en cas de régression

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_pre.py $MODEL_ARGS \
  --dataset redial \
  --num_train_epochs 1 --max_train_steps 5 \
  --gradient_accumulation_steps 1 \
  --per_device_train_batch_size 2 --per_device_eval_batch_size 2 \
  --num_warmup_steps 1 --max_length 200 --prompt_max_length 200 \
  --entity_max_length 32 --learning_rate 5e-4 --seed 42 --nei_mer
""",
    ],
    check=True,
)

### 2.3  Entraînement complet (commande README)

Comptez plusieurs heures sur une carte 24 Go ; la RTX 5090 dispose de 32 Go, vous pouvez donc augmenter les batch sizes si besoin.

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_pre.py $MODEL_ARGS \
  --dataset redial \
  --num_train_epochs 10 \
  --gradient_accumulation_steps 2 \
  --per_device_train_batch_size 32 \
  --per_device_eval_batch_size 128 \
  --num_warmup_steps 1389 \
  --max_length 200 --prompt_max_length 200 --entity_max_length 32 \
  --learning_rate 5e-4 --seed 42 --nei_mer
""",
    ],
    check=True,
)

## 3. Tâche B — Recommandation d'items sensible aux émotions (`train_rec.py`)

Fusionne l'inférence conversationnelle de UniCRS dans les données, prétraite une variante masquée,
puis entraîne la tête de recommandation.

### 3.1  Prétraitement : `process_mask.py` + fusion dans `redial_gen/`

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
cp -r data/emo_data/* data/redial/
cd data/redial && python process_mask.py
""",
    ],
    check=True,
)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
cp -r data/redial/* data/redial_gen/
cd data/redial_gen && python merge.py
""",
    ],
    check=True,
)

### 3.2  Smoke-test (5 étapes)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_rec.py $MODEL_ARGS \
  --dataset redial_gen \
  --n_prefix_rec 10 \
  --num_train_epochs 1 --max_train_steps 5 \
  --per_device_train_batch_size 2 --per_device_eval_batch_size 2 \
  --gradient_accumulation_steps 1 \
  --num_warmup_steps 1 \
  --context_max_length 200 --prompt_max_length 200 --entity_max_length 32 \
  --learning_rate 1e-4 --seed 8 \
  --like_score 2.0 --dislike_score 1.0 --notsay_score 0.5 \
  --weighted_loss --nei_mer --use_sentiment
""",
    ],
    check=True,
)

### 3.3  Entraînement complet (commande README)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_rec.py $MODEL_ARGS \
  --dataset redial_gen \
  --n_prefix_rec 10 \
  --num_train_epochs 5 \
  --per_device_train_batch_size 32 \
  --per_device_eval_batch_size 64 \
  --gradient_accumulation_steps 4 \
  --num_warmup_steps 530 \
  --context_max_length 200 --prompt_max_length 200 --entity_max_length 32 \
  --learning_rate 1e-4 --seed 8 \
  --like_score 2.0 --dislike_score 1.0 --notsay_score 0.5 \
  --weighted_loss --nei_mer --use_sentiment
""",
    ],
    check=True,
)

## 4. Tâche C — Génération de réponses alignées émotionnellement (backbone DialoGPT)

Affinage du générateur de réponses empathiques, puis inférence sur le split de test.

### 4.1  Prétraitement : fusion des sorties du recommandeur, filtrage des reviews IMDb, création des données empathiques

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
cd data/redial_gen && python merge_rec.py
""",
    ],
    check=True,
)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python imdb_review_entity_filter.py
""",
    ],
    check=True,
)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
cd data/redial && python process_empthetic.py
""",
    ],
    check=True,
)

### 4.2  Entraînement smoke-test (5 étapes)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_emp.py $MODEL_ARGS \
  --dataset redial \
  --num_train_epochs 1 --max_train_steps 5 \
  --gradient_accumulation_steps 1 \
  --ignore_pad_token_for_loss \
  --per_device_train_batch_size 2 --per_device_eval_batch_size 2 \
  --num_warmup_steps 1 \
  --context_max_length 150 --resp_max_length 150 \
  --learning_rate 1e-4
""",
    ],
    check=True,
)

### 4.3  Entraînement complet (commande README)

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python train_emp.py $MODEL_ARGS \
  --dataset redial \
  --num_train_epochs 15 \
  --gradient_accumulation_steps 1 \
  --ignore_pad_token_for_loss \
  --per_device_train_batch_size 20 \
  --per_device_eval_batch_size 128 \
  --num_warmup_steps 9965 \
  --context_max_length 150 --resp_max_length 150 \
  --learning_rate 1e-4
""",
    ],
    check=True,
)

### 4.4  Inférence avec les checkpoints publiés (`infer_emp.py`)

Cette étape utilise les checkpoints de `src_emo/data/saved/emp/` téléchargés depuis Drive.
Réduisez `--per_device_eval_batch_size` en cas d'erreur OOM.

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "-lc",
        """
set -euo pipefail
cd \"$SRC\"
python infer_emp.py $MODEL_ARGS \
  --dataset redial_gen \
  --split test \
  --per_device_eval_batch_size 384 \
  --context_max_length 150 --resp_max_length 150
""",
    ],
    check=True,
)